# Skin lesion generalization: Colab runner

Backup for when Kaggle's GPU queue is full. Results are saved to your Google Drive after every run, so you can stop at any time and press **Run all** again later: finished runs are skipped.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. When asked, allow Google Drive, then choose your `kaggle.json` file (Kaggle → Settings → API → Create New Token). The token stays in this Colab session; it is not saved to Drive or GitHub.

In [ ]:
# 1. Google Drive: results live here between sessions
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/skin-lesion-generalization'
os.makedirs(DRIVE + '/results', exist_ok=True)
os.makedirs(DRIVE + '/results_leaky', exist_ok=True)
!nvidia-smi -L

In [ ]:
# 2. Your Kaggle token (choose kaggle.json when the file picker opens)
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('token ready')

In [ ]:
%%bash
# 3. Download the three datasets (about 15 minutes) and the Version 1 results
pip install -q kaggle timm grad-cam onnx onnxruntime
mkdir -p /content/input /content/work
cd /content/input
[ -d isic-2019 ] || kaggle datasets download -q -d andrewmvd/isic-2019 -p isic-2019 --unzip
[ -d skin-cancer-mnist-ham10000 ] || kaggle datasets download -q -d kmader/skin-cancer-mnist-ham10000 -p skin-cancer-mnist-ham10000 --unzip
[ -d skin-cancer ] || kaggle datasets download -q -d mahdavi1202/skin-cancer -p skin-cancer --unzip
if [ ! -d /content/v1 ]; then kaggle kernels output muhammadmozzam/skin-lesion-generalization -p /content/v1 || true; fi
ls /content/input; du -sh /content/input/*

In [ ]:
%%bash
# 4. Code from GitHub; Version 1 runs copied to Drive once (finished runs are not repeated)
rm -rf /content/work/skin-lesion-generalization
git clone -q https://github.com/muhammadmozzam/skin-lesion-generalization /content/work/skin-lesion-generalization
git -C /content/work/skin-lesion-generalization log -1 --format=%H
D=/content/drive/MyDrive/skin-lesion-generalization
for d in $(find /content/v1 -maxdepth 3 -type d -name results 2>/dev/null); do cp -rn $d/. $D/results/; done
ln -sfn $D/results /content/work/results
ln -sfn $D/results_leaky /content/work/results_leaky
ls $D/results

In [ ]:
# 5. Run (the same stages as on Kaggle). Safe to stop and re-run: finished runs are skipped.
import os
os.environ['SLG_WORK'] = '/content/work'
os.environ['SLG_INPUT'] = '/content/input'
R = 'python /content/work/skin-lesion-generalization/run_kaggle.py'
!{R} --stage prep
!{R} --stage train --seeds 0,1
!{R} --stage leaky --seeds 0
!{R} --stage summary
!{R} --stage calibrate
!{R} --stage pack && cp /content/work/results.zip /content/drive/MyDrive/skin-lesion-generalization/